# d₄₀₋₁₇ Distance Metric Analysis

**What this notebook does:**
- Computes the inter-residue distance **d₄₀₋₁₇** (ASP40–ASP17 distance) from GROMACS
  distance CSV files for all available simulations
- Converts GROMACS distances from nm to Å (1 nm = 10 Å)
- Plots the distance trace for each simulation, split into **Phase 1** and **Phase 2**
  using per-simulation phase boundaries stored in `PHASE_LIMIT_DICO`
- Saves a zoomed view showing only Phase 2
- Computes and exports descriptive statistics to CSV

**Background:**
d₄₀₋₁₇ measures the distance between ASP40 (chain A) and ASP17 (chain B, residue index
shifted by +99 → column `116_139` in the distance files).

**Sections:**
1. Install & import libraries
2. Configuration (paths, phase limits)
3. Helper function: `cal_d_40_17()`
4. Discover distance files
5. Plot all simulations (Phase 1 + Phase 2)
6. Plot zoomed view (Phase 2 only)
7. Compute and export statistics


In [ ]:
# 1. Imports
import os
import math
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# 2. Configuration — adapt these paths to your environment
ROOTDIR    = '/content/drive/MyDrive/M1_STAGE/Data/simulations_1HSI/'
OUTPUT_DIR = '/content/drive/MyDrive/M1_STAGE/Manips/'

os.chdir(ROOTDIR)

# Phase boundaries: frame index where Phase 1 ends and Phase 2 begins
PHASE_LIMIT_DICO = {
    'V7':  198,
    'V8':  180,
    'V21': 155,
    'V12': 35,
    'V11': 1000,
    'V1':  517,
}

In [ ]:
# 3. Helper function — extract d₄₀₋₁₇ from a distance DataFrame
def cal_d_40_17(distance_df: pd.DataFrame) -> pd.Series:
    """
    Return the d₄₀₋₁₇ time series from a GROMACS distance file.
    Column '116_139' corresponds to chain B residues (ASP40 chB → index shifted).
    """
    return distance_df['116_139']

In [ ]:
# 4. Discover all distance files (files starting with 'distance_')
distance_files_paths = []
for subdir, _, files in os.walk(ROOTDIR):
    for f in files:
        if f.startswith('distance_'):
            distance_files_paths.append(os.path.join(subdir, f))

distance_files_paths.sort()
print(f'Found {len(distance_files_paths)} distance file(s):')
for p in distance_files_paths:
    print(' ', os.path.basename(p))

In [ ]:
# 5. Plot d₄₀₋₁₇ for all simulations — Phase 1 (pink) vs Phase 2 (cyan)
num_files = len(distance_files_paths)
num_cols  = min(num_files, 3)
num_rows  = math.ceil(num_files / num_cols)

fig, axes = plt.subplots(num_rows, num_cols,
                          figsize=(num_cols * 8, num_rows * 6), squeeze=False)
axes = axes.flatten()
fig.suptitle(r'd$_{17-40}$ Analysis: Phase 1 vs Phase 2', fontsize=22, y=0.92)
line_ph1 = line_ph2 = None

for plot_idx, distance_file in enumerate(distance_files_paths):
    ax = axes[plot_idx]
    sim_name = os.path.basename(distance_file).replace('distance_', '').replace('.csv', '')

    df = pd.read_csv(distance_file) * 10          # nm → Å
    df.index.name = 'step'
    d_series = cal_d_40_17(df)
    phase_limit = PHASE_LIMIT_DICO.get(sim_name, len(d_series))

    line_ph1, = ax.plot(d_series.index[:phase_limit],  d_series[:phase_limit],
                         color='pink', linewidth=1.5)
    line_ph2, = ax.plot(d_series.index[phase_limit:],  d_series[phase_limit:],
                         color='cyan', linewidth=1.5)
    ax.set_title(f'Simulation: {sim_name}')
    ax.set_xlabel('Step')
    ax.set_ylabel(r'd$_{40-17}$ (Å)')
    ax.set_ylim(8, 25)
    ax.grid(True, alpha=0.3)

# Hide unused axes
for j in range(plot_idx + 1, len(axes)):
    fig.delaxes(axes[j])

if line_ph1 and line_ph2:
    fig.legend(handles=[line_ph1, line_ph2], labels=['Phase 1', 'Phase 2'],
               loc='center', bbox_to_anchor=(0.92, 0.5),
               fontsize=14, frameon=True)

plt.tight_layout(rect=[0, 0, 0.85, 0.92])
plt.show()
fig.savefig(os.path.join(OUTPUT_DIR, 'Figures/d_40_17.png'), bbox_inches='tight', dpi=300)

In [ ]:
# 6. Zoomed plot — Phase 2 only
fig, axes = plt.subplots(num_rows, num_cols,
                          figsize=(num_cols * 8, num_rows * 6), squeeze=False)
axes = axes.flatten()
fig.suptitle(r'd$_{17-40}$ — Phase 2 (zoomed view)', fontsize=22, y=0.92)

for plot_idx, distance_file in enumerate(distance_files_paths):
    ax = axes[plot_idx]
    sim_name = os.path.basename(distance_file).replace('distance_', '').replace('.csv', '')

    df = pd.read_csv(distance_file) * 10
    df.index.name = 'step'
    d_series = cal_d_40_17(df)
    phase_limit = PHASE_LIMIT_DICO.get(sim_name, len(d_series))

    ax.plot(d_series.index[phase_limit:], d_series[phase_limit:],
            color='cyan', linewidth=1.5)
    ax.set_title(f'Simulation: {sim_name}')
    ax.set_xlabel('Step')
    ax.set_ylabel(r'd$_{40-17}$ (Å)')
    ax.grid(True, alpha=0.3)

for j in range(plot_idx + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout(rect=[0, 0, 0.85, 0.92])
plt.show()
fig.savefig(os.path.join(OUTPUT_DIR, 'Figures/d_40_17_zoomed.png'), bbox_inches='tight', dpi=300)

In [ ]:
# 7. Descriptive statistics per simulation — exported to CSV
summary_stats_dict = {}

for distance_file in distance_files_paths:
    sim_name = os.path.basename(distance_file).replace('distance_', '').replace('.csv', '')
    df = pd.read_csv(distance_file) * 10
    df.index.name = 'step'
    d_series = cal_d_40_17(df)
    summary_stats_dict[sim_name] = d_series.describe()

summary_stats_df = pd.DataFrame(summary_stats_dict)
summary_stats_df.to_csv(
    os.path.join(OUTPUT_DIR, 'Tables/d_40_17_stats.csv'),
    index=True, header=True, decimal='.', float_format='%.3f'
)
summary_stats_df